In [1]:
from datetime import datetime
import pandas as pd
from meteostat import Point, Daily

def fetch_weather_data():
    # 1. Zeitraum festlegen (hier exemplarisch das Jahr 2023)
    # Für euer finales Projekt könnt ihr das z.B. auf 2020 bis 2023 ausweiten
    start = datetime(2023, 1, 1)
    end = datetime(2023, 12, 31)

    # 2. Unsere drei Vergleichs-Standorte definieren 
    # Parameter: Point(Breitengrad, Längengrad, Höhe in Metern)
    locations = {
        "München": Point(48.1351, 11.5820, 520),
        "Essen": Point(51.4556, 7.0116, 116),
        "Schleswig": Point(54.5167, 9.5667, 15)
    }

    all_data = []

    print("Starte Wetter-Download...")

    # 3. Schleife über unsere Städte
    for city, point in locations.items():
        print(f"Lade Daten für {city}...")
        
        # Daily() ruft die Tagesmittelwerte ab
        data = Daily(point, start, end)
        data = data.fetch()
        
        # Wenn Daten gefunden wurden, bereiten wir sie auf
        if not data.empty:
            # Meteostat hat das Datum als Index, wir machen es zu einer normalen Spalte
            data = data.reset_index()
            
            # Wir fügen die Stadt als Spalte hinzu, damit wir später wissen, 
            # welche Zeile zu wem gehört
            data['Stadt'] = city
            
            all_data.append(data)
        else:
            print(f"Achtung: Keine Daten für {city} gefunden.")

    # 4. Alle einzelnen DataFrames zu einem großen zusammenkleben
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        
        # 5. Wir filtern nur die Spalten heraus, die wir wirklich brauchen
        # tavg = Durchschnittstemperatur (°C)
        # prcp = Niederschlag (mm)
        # wspd = Windgeschwindigkeit (km/h)
        columns_to_keep = ['time', 'Stadt', 'tavg', 'prcp', 'wspd']
        final_df = final_df[columns_to_keep]
        
        # Spalten für die bessere Lesbarkeit umbenennen
        final_df.rename(columns={
            'time': 'Datum',
            'tavg': 'Temperatur_Avg',
            'prcp': 'Niederschlag_mm',
            'wspd': 'Wind_kmh'
        }, inplace=True)
        
        print("\nDownload abgeschlossen! Datenaufbereitung erfolgreich.")
        return final_df
    else:
        return None

# ==========================================
# TESTLAUF
# ==========================================
if __name__ == "__main__":
    wetter_df = fetch_weather_data()
    
    if wetter_df is not None:
        print("\nHier sind die ersten 5 Zeilen unseres neuen Datensatzes:")
        display(wetter_df.head())
        
        print("\nUnd hier prüfen wir, ob alle 3 Städte drin sind:")
        print(wetter_df['Stadt'].value_counts())
        
        # So speichert ihr das direkt für Streamlit ab:
        # wetter_df.to_csv("data/processed/wetter_deepdive_2023.csv", index=False)

ImportError: cannot import name 'Daily' from 'meteostat' (C:\Users\fhasd\AppData\Local\Programs\Python\Python312\Lib\site-packages\meteostat\__init__.py)